<a href="https://colab.research.google.com/github/Mohammed414/ModalPINN2.0/blob/analysis/a03-closeout-and-workspace-sync/notebooks/karman_prior_derivation_and_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The tap-derived Kármán prior

### An executable companion to the dissertation

This notebook builds, runs and tests the analytical wake prior used by the
**V1-only radial-trust ModalPINN** experiment. Everything here runs from a
cold start in Google Colab in under a minute. There is nothing to install and
nothing to download by hand.

**Run it:** `Runtime → Run all`, or press each ▶ in turn. Cells with sliders can
be re-run with different values — the physics recomputes each time.

---

**What the prior is.** The reported experiment gives the neural network a
closed-form Kármán vortex street as a starting point for the first harmonic of
the transverse velocity, then lets the network correct it within a bounded
trust region. This notebook derives that street, plots it, and reproduces the
exact gate and blend used during training.

**Where the information comes from.** Every parameter is fixed by the same 32
cylinder-pressure histories the network itself receives, plus the known
geometry and classical vortex-street relations. **No interior CFD velocity is
fitted here.** The CFD field is used only afterwards, outside this notebook,
to score the reconstruction.

**No TensorFlow, no training, no GPU.** NumPy, SciPy and Matplotlib only.

## 0. Setup

Fetches three small files from the public repository: the frozen 32-tap
parameter archive, and the two modules that define the vortex street. Nothing
is installed — Colab already has NumPy, SciPy and Matplotlib.

In [ ]:
# --- fetch the frozen prior and the code that reads it -----------------------
import urllib.request, pathlib, sys

RAW = "https://raw.githubusercontent.com/Mohammed414/ModalPINN2.0/analysis/a03-closeout-and-workspace-sync"
FILES = {
    "street_prior_Ntap32.npz": "results/data/geometry/street_prior_Ntap32.npz",
    "street_prior.py":         "results/code/vendor/street_prior.py",
    "text_flow.py":            "results/code/vendor/text_flow.py",
}
for name, path in FILES.items():
    if not pathlib.Path(name).exists():
        urllib.request.urlretrieve(f"{RAW}/{path}", name)
    print(f"  {name:28s} {pathlib.Path(name).stat().st_size/1024:7.1f} KB")

sys.path.insert(0, ".")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from street_prior import Street, cf_modes_uv, HA_RATIO, NU, R_C

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 10, "axes.titlesize": 11,
    "axes.labelsize": 10, "axes.spines.top": False,
    "axes.spines.right": False, "savefig.bbox": "tight",
})

# --- the frozen archive ------------------------------------------------------
archive = np.load("street_prior_Ntap32.npz")
NAMES = ("Gamma", "Uc", "xf", "r0", "omega", "phase",
         "amp_scale", "scale_p", "ramp", "delta")
prm = {n: float(archive[n]) for n in NAMES}

print("\nParameters derived from the 32 pressure taps:")
MEANING = {
    "Gamma": "circulation of each vortex",
    "Uc":    "convection speed of the street",
    "xf":    "formation position",
    "r0":    "initial Lamb-Oseen core radius",
    "omega": "shedding angular frequency",
    "phase": "phase offset",
    "amp_scale": "closed-form to numerical amplitude calibration",
    "scale_p":   "pressure scaling",
    "ramp":      "formation envelope width",
    "delta":     "smoothing length",
}
for n in NAMES:
    print(f"  {n:>10s} = {prm[n]: .6f}    {MEANING[n]}")

## 1. Physical model

Two staggered vortex rows are placed at $y=\pm h/2$. Their streamwise spacing is $a$, the lower row is offset by $a/2$, and the classical stable spacing ratio is

$$\frac{h}{a}=0.281, \qquad a=\frac{2\pi U_c}{\omega_0}. $$

Each vortex is regularised as a Lamb–Oseen core. For a vortex centred at $\mathbf{x}_v$, with $\Delta x=x-x_v$, $\Delta y=y-y_v$ and $r^2=\Delta x^2+\Delta y^2$, the induced Cartesian velocity is

$$u=-\frac{\Gamma\,\Delta y}{2\pi r^2}\left(1-e^{-r^2/r_c^2}\right), \qquad v=\frac{\Gamma\,\Delta x}{2\pi r^2}\left(1-e^{-r^2/r_c^2}\right).$$

The core grows downstream according to

$$r_c^2(x_v)=r_0^2+\frac{4\nu\max(x_v-x_f,0)}{U_c}, \qquad \nu=1/Re.$$

The numerical reference street also includes Milne–Thomson image vortices so that the cylinder is a streamline, a potential-flow dipole for the mean field, and a smooth formation envelope $[1+\tanh((x-x_f)/\delta_x)]/2$.

## 2. How the 32 taps determine the archive

1. The tap pressures are integrated around the cylinder to obtain pressure-drag and lift histories.
2. A sinusoid is fitted to lift to estimate $\omega_0$.
3. Each tap signal is projected onto $\{1,\cos(\omega_0t),\sin(\omega_0t)\}$, giving its first pressure harmonic.
4. Circulation $\Gamma$ is obtained by solving the Kármán drag relation together with

$$U_c=1-\frac{\Gamma}{\sqrt{8}a}, \qquad a=\frac{2\pi U_c}{\omega_0}.$$

5. Formation position $x_f$, initial core radius $r_0$, and phase are chosen by matching the image-street surface-pressure harmonic to the measured tap harmonic.
6. The differentiable closed-form harmonic is amplitude- and phase-aligned to the numerical analytical street at wake points. This is street-to-street calibration; it does not use the CFD interior field.

The original circulation calculation used $C_{D,\mathrm{total}}\simeq C_{D,p}/0.75$. A later full-wall stress integration measured $\overline C_{D,p}=0.9890$, $\overline C_{D,\nu}=0.3400$, and $\overline C_D=1.3291$, so the measured pressure fraction is **74.42%**. The rounded 75% assumption differs by 0.58 percentage points.

### Derived quantities, and the checks that must pass

The next cell asserts two things. If either fails the notebook stops, so a
green run is itself the verification.

In [ ]:
a = 2 * np.pi * prm["Uc"] / prm["omega"]
h = HA_RATIO * a
period = 2 * np.pi / prm["omega"]
street = Street(prm["Gamma"], prm["Uc"], x_f=prm["xf"], r0=prm["r0"],
                phase=prm["phase"], omega=prm["omega"], ramp=prm["ramp"])

print(f"Shedding period      T   = {period:.4f}")
print(f"Streamwise spacing   a   = {a:.4f} D")
print(f"Row separation       h   = {h:.4f} D")
print(f"Spacing ratio        h/a = {h/a:.4f}   (Karman's stable value: 0.281)")
print(f"Pressure drag        CDp = {float(archive['CD_pressure']):.4f}")
print(f"Tap first-harmonic correlation      = {float(archive['tap_p1_corr']):.4f}")
print(f"Closed-form vs numerical correlation = {float(archive['cf_corr_vs_numeric']):.4f}")

assert abs(h / a - 0.281) < 1e-12, "spacing ratio is not the Karman value"
assert float(archive["cf_corr_vs_numeric"]) > 0.95, "closed form disagrees with the numerical street"
print("\nBoth checks passed.")

## 3. Run the vortex street

The `Street` class below is the same one used when the prior was derived. Red
and blue markers are the opposite-signed staggered vortices; colour and
streamlines show the analytical velocity field.

**Try it:** move the phase slider and re-run to advance the street in time.
One full period is `T` printed above.

In [ ]:
phase_fraction = 0  # @param {type:"slider", min:0, max:1, step:0.05}
t_show = phase_fraction * period

xg = np.linspace(-1.0, 8.0, 220)
yg = np.linspace(-2.3, 2.3, 120)
X, Y = np.meshgrid(xg, yg)
u, v = street.velocity(np.column_stack((X.ravel(), Y.ravel())), t=t_show)
U, V = u.reshape(X.shape), v.reshape(X.shape)
speed = np.sqrt(U**2 + V**2)
inside = X**2 + Y**2 <= R_C**2
U, V = np.ma.array(U, mask=inside), np.ma.array(V, mask=inside)
speed = np.ma.array(speed, mask=inside)
upper, lower = street._vortex_positions(t=t_show)

fig, ax = plt.subplots(figsize=(10.5, 4.2), constrained_layout=True)
levels = np.linspace(0.0, np.nanpercentile(speed.compressed(), 98), 24)
contour = ax.contourf(X, Y, speed, levels=levels, cmap="viridis", extend="max")
ax.streamplot(xg, yg, U, V, color="white", density=1.15, linewidth=0.45, arrowsize=0.65)
su = upper[(upper[:, 0] >= -1.0) & (upper[:, 0] <= 8.0)]
sl = lower[(lower[:, 0] >= -1.0) & (lower[:, 0] <= 8.0)]
ax.scatter(su[:, 0], su[:, 1], s=42, c="#C43C39", edgecolor="white",
           linewidth=0.7, label=r"upper row, $-\Gamma$", zorder=5)
ax.scatter(sl[:, 0], sl[:, 1], s=42, c="#2B6CB0", edgecolor="white",
           linewidth=0.7, label=r"lower row, $+\Gamma$", zorder=5)
ax.add_patch(Circle((0, 0), R_C, facecolor="#263238", edgecolor="white", linewidth=1.0, zorder=6))
ax.set(xlim=(-1, 8), ylim=(-2.3, 2.3), xlabel=r"$x/D$", ylabel=r"$y/D$",
       title=f"Lamb-Oseen street from the tap-derived parameters, t = {t_show:.2f}")
ax.set_aspect("equal")
ax.legend(loc="upper right", frameon=True, ncol=2)
fig.colorbar(contour, ax=ax, pad=0.015).set_label(r"speed $|\mathbf{u}|/U_\infty$")
plt.show()

## 4. Differentiable first harmonic used by ModalPINN

For row $j$, the closed-form contribution at harmonic $k$ is

$$B_{j,k}=s_j\frac{\Gamma}{2a}\exp\!\left[-\frac{2\pi k}{a}\operatorname{softabs}(y-y_j)\right]\exp\!\left[-\frac{(\pi k)^2r_c^2(x)}{a^2}\right]\exp\!\left[-i\frac{2\pi k(x-x_{0,j})}{a}-ik\phi\right].$$

The transverse mode is $S_{v,k}=i\,f(x)\sum_j B_{j,k}$, multiplied by the saved street-to-street amplitude calibration and converted to the one-sided ModalPINN convention. The reported experiment uses **only** $S_{v,1}$ in its trust region. The full numerical street above helps derive and visualise the prior, but it is not pasted wholesale into the network.

### The trust gate

The gate $W(x,y)$ decides **where** the prior is allowed to act. It is 1 deep
in the wake, 0 near the cylinder and outside the wake, with a smooth
transition. The reported runs use `xstart = 3.0`, `ymax = 2.0`.

**Try it:** move the sliders and re-run to see the trusted region change shape.

In [ ]:
xstart = 3.0   # @param {type:"slider", min:0.5, max:6.0, step:0.1}
xwidth = 0.30  # @param {type:"slider", min:0.05, max:2.0, step:0.05}
ymax   = 2.0   # @param {type:"slider", min:0.5, max:3.0, step:0.1}
ywidth = 0.20  # @param {type:"slider", min:0.05, max:1.5, step:0.05}

def smootherstep01(z):
    z = np.clip(z, 0.0, 1.0)
    return z**3 * (z * (z * 6.0 - 15.0) + 10.0)

def v1_trust_gate(x, y, xstart=3.0, xwidth=0.30, ymax=2.0, ywidth=0.20):
    wx = smootherstep01((x - (xstart - xwidth)) / xwidth)
    wy_hi = 1.0 - smootherstep01((y - ymax) / ywidth)
    wy_lo = 1.0 - smootherstep01((-y - ymax) / ywidth)
    return wx * wy_hi * wy_lo

us, vs = cf_modes_uv(X.ravel(), Y.ravel(), prm, nk=1)
S_v1 = (2.0 * prm["amp_scale"] * vs[0]).reshape(X.shape)
W = v1_trust_gate(X, Y, xstart, xwidth, ymax, ywidth)

fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.5), constrained_layout=True,
                         sharex=True, sharey=True)
items = [
    (np.abs(S_v1), r"Prior amplitude $|S_{v,1}|$", "magma"),
    (W, r"Trust gate $W(x,y)$", "Blues"),
    (W * np.abs(S_v1), "Prior inside the trusted region", "magma"),
]
for ax, (field, title, cmap) in zip(axes, items):
    im = ax.pcolormesh(X, Y, field, shading="auto", cmap=cmap)
    ax.add_patch(Circle((0, 0), R_C, facecolor="#263238", edgecolor="white", linewidth=0.8))
    ax.axvline(xstart, color="white", linewidth=0.8, linestyle="--")
    ax.set(xlim=(-1, 8), ylim=(-2.3, 2.3), xlabel=r"$x/D$", title=title)
    ax.set_aspect("equal")
    fig.colorbar(im, ax=ax, pad=0.02, shrink=0.82)
axes[0].set_ylabel(r"$y/D$")
plt.show()

frac = float((W > 0.5).mean())
print(f"The gate is open over {frac*100:.1f}% of this window.")
assert W.min() >= 0.0 and W.max() <= 1.0 + 1e-14, "gate left [0, 1]"

## 5. Exact reported hybrid ansatz

Let $z(x,y)$ be the free complex output of the transverse-velocity network, $f_{BC}$ the hard cylinder no-slip factor, and $W(x,y)$ the gate plotted above. The reported model evaluates

$$\widehat v_1=f_{BC}\left[(1-W)z+W\left(S_{v,1}+\rho_{\mathrm{tr}}|S_{v,1}|_\varepsilon\frac{z}{\sqrt{1+|z|^2}}\right)\right], \qquad \rho_{\mathrm{tr}}=0.60.$$

The smooth magnitude $|S|_\varepsilon=\sqrt{(\Re S)^2+(\Im S)^2+\varepsilon^2}-\varepsilon$ avoids a non-smooth absolute value inside a field differentiated by the PDE loss. In the trusted core, the correction magnitude is strictly less than $0.60|S_{v,1}|_\varepsilon$, so a non-zero prior cannot collapse exactly to zero. Outside the gate, $\widehat v_1=f_{BC}z$ is the ordinary ModalPINN output.

This distinction matters: the earlier R9 `--TrustStreet` prototype applied the street to $u$, $v$, and $p$ for every $k\ge1$. The dissertation results instead use the later `--V1RadialTrust` option described here.

### What the trust bound actually guarantees

The blend is built so the network can reshape the prior but never erase it.
Whatever the network outputs — however large — the correction term is bounded
by $\rho_{\mathrm{tr}}|S_{v,1}|$, because $|z|/\sqrt{1+|z|^2} < 1$ always.

The cell below demonstrates that numerically: it pushes random network outputs
through the blend at increasing magnitude and checks the bound holds.

**Try it:** raise $\rho_{\mathrm{tr}}$ towards 1 to loosen the prior, or lower
it towards 0 to pin the reconstruction to the analytical street.

In [ ]:
rho_tr = 0.60  # @param {type:"slider", min:0.0, max:1.0, step:0.05}
eps = 1e-8
rng = np.random.default_rng(0)

def blend(S, z, rho=rho_tr):
    """The reported V1 radial-trust blend."""
    S_mag = np.sqrt(S.real**2 + S.imag**2 + eps**2) - eps
    return S + rho * S_mag * z / np.sqrt(1.0 + np.abs(z)**2)

S_probe = S_v1[(W > 0.99)]
print(f"{'network |z|':>14s}   {'max |correction| / |S|':>24s}   within bound?")
for scale in (0.1, 1.0, 10.0, 1000.0, 1e6):
    z = scale * (rng.standard_normal(S_probe.size) + 1j * rng.standard_normal(S_probe.size))
    dev = np.abs(blend(S_probe, z) - S_probe) / (np.abs(S_probe) + 1e-30)
    print(f"{scale:14.1e}   {dev.max():24.6f}   {'yes' if dev.max() <= rho_tr + 1e-9 else 'NO'}")
    assert dev.max() <= rho_tr + 1e-9, "trust bound violated"

print(f"\nThe correction can never exceed {rho_tr:.2f} x the prior magnitude,")
print("so inside the gate a non-zero prior cannot collapse to zero.")
print("This is what stops the network from choosing the trivial quiet solution.")

## 6. What this notebook shows — and what it does not

**It shows** that the frozen archive is readable, that the physical street runs
with no neural network anywhere, that the closed-form harmonic agrees with the
numerical street above 0.95 correlation, and that the trust gate and blend
match the reported run configuration exactly.

**It does not** validate reconstruction accuracy against CFD. Those comparisons
are made by the shared evaluator and collected in the results bundle.

### Where everything else lives

| | |
|---|---|
| Full numerical results, filterable | [`results/all_results.xlsx`](https://github.com/Mohammed414/ModalPINN2.0/blob/analysis/a03-closeout-and-workspace-sync/results/all_results.xlsx) |
| Automated provenance and consistency checks | [`results/code/verify.py`](https://github.com/Mohammed414/ModalPINN2.0/blob/analysis/a03-closeout-and-workspace-sync/results/code/verify.py) |
| Shared region and metric definitions | [`results/code/evaluate_common.py`](https://github.com/Mohammed414/ModalPINN2.0/blob/analysis/a03-closeout-and-workspace-sync/results/code/evaluate_common.py) |
| Tap-to-parameter chain and closed-form modes | [`street_prior.py`](https://github.com/Mohammed414/ModalPINN2.0/blob/analysis/a03-closeout-and-workspace-sync/results/code/vendor/street_prior.py) |
| Boudina CFD dataset | [Zenodo record 5039610](https://zenodo.org/records/5039610) |